# 06. 단계적 공개와 설계 비교

단계적 공개는 먼저 짧은 목록을 보고, 필요한 문서 ID를 고른 뒤, 해당 원문만 모델에 전달하는 방식입니다.

## 실행 환경 준비

첫 코드 셀은 노트북에서 사용할 패키지 버전을 맞춥니다. 설치가 끝나면 이어지는 셀을 순서대로 실행합니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
# model은 이후 셀에서 공통으로 사용할 LangChain 채팅 모델 객체입니다.
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
import os

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model("openai:gpt-5.4-mini", reasoning_effort="none", max_tokens=300)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 목록과 원문을 분리합니다

목록은 선택을 돕고, 답은 선택한 원문에서 만듭니다.

In [3]:
# documents는 ID·제목·원문을 가진 문서 목록입니다.
# 리스트 컴프리헨션은 선택 화면에 필요한 ID와 제목만 catalog에 반환합니다.
documents = [
    {"id": "E01-p2", "title": "이용 시간", "text": "월요일 휴관"},
    {"id": "E01-p3", "title": "대출 규정",
     "text": "7권을 14일 대출. 예약이 없으면 1회 7일 연장."},
    {"id": "E01-p5", "title": "주차 안내", "text": "90분 무료"},
]
catalog = [{"id": d["id"], "title": d["title"]} for d in documents]
print(catalog)

[{'id': 'E01-p2', 'title': '이용 시간'}, {'id': 'E01-p3', 'title': '대출 규정'}, {'id': 'E01-p5', 'title': '주차 안내'}]


## 3. 필요한 ID의 원문만 공개합니다

이 예제에서는 학생이 `E01-p3`을 직접 선택합니다.

In [4]:
# selected_id는 질문에 맞춰 고른 문서 ID이고, next()는 첫 일치 문서를 반환합니다.
# full·selected·compressed의 문자 수를 비교해 공개 범위 차이를 확인합니다.
selected_id = "E01-p3"
selected = next(d for d in documents if d["id"] == selected_id)
full = "\n".join(d["text"] for d in documents)
compressed = "7권, 14일, 예약 없으면 1회 7일 연장 [E01 p.3]"
print("full 문자 수:", len(full))
print("selected 문자 수:", len(selected["text"]))
print("compressed 문자 수:", len(compressed))

full 문자 수: 43
selected 문자 수: 29
compressed 문자 수: 34


## 4. 선택한 원문으로 답을 생성합니다

In [5]:
# selected의 원문만 f-string에 넣어 모델에 전달할 prompt를 만듭니다.
# 질문과 근거를 한 문자열에 두되 출처 표기를 함께 남깁니다.
prompt = f"""근거 [E01 p.3]: {selected["text"]}
질문: 대출과 연장 조건은?"""

In [6]:
# messages는 역할과 내용을 짝지은 목록이며 system 지시를 user 질문보다 먼저 둡니다.
# invoke()는 모델을 한 번 호출하고, 반환된 AIMessage에서 text와 사용량을 읽습니다.
messages = [
    ("system", "근거의 조건과 출처를 빠뜨리지 마세요."),
    ("user", prompt),
]
response = model.invoke(messages)
# usage_metadata는 입력·출력 토큰 수를 담지만 제공되지 않을 가능성도 검사합니다.
usage = response.usage_metadata
if usage is None:
    raise RuntimeError("모델 사용량 메타데이터가 없습니다.")
print(response.text)
print("input_tokens:", usage.get("input_tokens"))
print("output_tokens:", usage.get("output_tokens"))

대출과 연장 조건은 다음과 같습니다.

- **대출**: 1회에 **7권까지**, **14일** 대출
- **연장**: **예약이 없을 경우** **1회에 한해 7일 연장 가능**

**근거**: [E01 p.3]
input_tokens: 67
output_tokens: 74


## 정리

목록, 선택, 원문 읽기를 순서대로 수행합니다. 토큰 감소와 필요한 조건의 보존 여부를 함께 비교합니다.

## 실패 재현: 질문과 다른 문서를 선택합니다

대출 질문에 주차 문서를 선택하면 입력은 짧아져도 필요한 대출 근거가 사라집니다.

In [7]:
# selected_id는 실제 선택값, required_id는 질문에 필요한 문서 ID입니다.
# 두 값이 다르면 잘못된 문서 선택이므로 ValueError를 발생시킵니다.
def require_document(selected_id: str, required_id: str) -> None:
    if selected_id != required_id:
        raise ValueError(f"필요한 문서 {required_id}가 선택되지 않았습니다.")

try:
    require_document("E01-p5", "E01-p3")
except ValueError as error:
    print("관찰한 실패:", error)
else:
    raise AssertionError("문서 선택 오류가 발생해야 합니다.")

관찰한 실패: 필요한 문서 E01-p3가 선택되지 않았습니다.


## 수정: 대출 규정 문서를 선택합니다

질문에 필요한 `E01-p3`을 선택하고 해당 원문을 확인합니다.

In [8]:
# fixed_id를 필요한 대출 규정 ID로 고친 뒤 선택 검사를 다시 실행합니다.
# next()로 실제 문서를 찾아 제목도 대출 규정인지 확인합니다.
fixed_id = "E01-p3"
require_document(fixed_id, "E01-p3")
fixed_document = next(d for d in documents if d["id"] == fixed_id)
assert fixed_document["title"] == "대출 규정"
print("수정 후 선택:", fixed_document["id"], fixed_document["title"])

수정 후 선택: E01-p3 대출 규정
